# 00 · Bring your agent

The harness improves **one thing**: the text that defines your agent's behavior (a system prompt,
an `AGENTS.md`, a `SKILL.md`). It keeps that text in the MLflow Prompt Registry, hands the chosen
version to your agent on every call, and records each call as a trace.

This notebook wires up the example agent, an Acme Cloud support agent with a `lookup_policy`
tool, and shows the two pieces everything else builds on: **a versioned prompt** and **one trace
per request**. Your own agent plugs in the same way through `.env`:

| `SI_AGENT` | what you provide |
|---|---|
| `python` | `def answer(question, instructions) -> str` (any SDK) |
| `http` | a service that takes `{"input", "instructions"}` |
| `command` | a CLI, with `{input}` and `{instructions_file}` filled in |
| `litellm` | nothing: a chat model with the instructions as its system prompt |
| `openclaw` | OpenClaw, with the instructions as `AGENTS.md` (isolated profile) |

The notebooks use their own store in `.selfimprove/notebooks/`. This first one starts it fresh.

In [24]:
import _nb as nb

experiment_id = nb.setup(fresh=True)

2026/10/28 12:18:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 12:18:42 INFO mlflow.store.db.utils: Updating database tables


env file      /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.env
tracking      sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.selfimprove/notebooks/mlflow.db   experiment walkthrough
agent         python: examples.support_agent:answer
instructions  prompts:/walkthrough-instructions  (seed /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/examples/instructions.md)
judge         anthropic:/claude-opus-5-5  (credentials found)
first pass    -
improver      analyst   auto-promote False
mode          live


**Under the hood:** `nb.setup()` calls the harness's `traces.setup()`, which points MLflow at the
notebooks' SQLite file and selects the `walkthrough` experiment. Because the agent is the built-in
example, it also turns on LiteLLM autologging, so live LLM calls become spans without any tracing code
in the agent.

**MLflow in this step**

* `mlflow.set_tracking_uri("sqlite:///.../mlflow.db")`: every later call reads and writes this store
* `mlflow.create_experiment("walkthrough", artifact_location=...)`: the first time only, so artifacts
  land in `.selfimprove/notebooks/artifacts`, not in `./mlruns`
* `mlflow.set_experiment("walkthrough")`: new traces and runs go to this experiment
* `mlflow.litellm.autolog()`: records each LiteLLM call as a span

In [25]:
import mlflow

exp = mlflow.get_experiment(experiment_id)
print("tracking uri ", mlflow.get_tracking_uri())
print("experiment   ", exp.name, "id", exp.experiment_id)
print("artifacts    ", exp.artifact_location)

tracking uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.selfimprove/notebooks/mlflow.db
experiment    walkthrough id 1
artifacts     file:///Users/jon/Documents/Claude/ODSC%20West%202026/self-improving-agent/harness/.selfimprove/notebooks/artifacts


## The contract

`examples/support_agent.py:answer` is all the harness needs from the example agent. When the
agent calls `lookup_policy`, that call becomes a TOOL span, and scorers use it to check whether the agent looked before it
answered.

In [26]:
import inspect
import examples.support_agent as agent

print(inspect.getsource(agent.answer))

def answer(question: str, instructions: str, workspace: Path | None = None) -> str:
    from selfimprove import models

    if os.getenv("SI_EXAMPLE_OFFLINE") == "1" or not models.has_credentials(MODEL):
        return answer_offline(question, instructions, workspace)
    messages = [{"role": "system", "content": instructions}, {"role": "user", "content": question}]
    for _ in range(5):
        msg = models.chat(MODEL, messages, tools=[LOOKUP_TOOL], max_tokens=1500)
        if not msg.tool_calls:
            return msg.content or ""
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            topic = json.loads(call.function.arguments or "{}").get("topic", "")
            try:
                result = lookup_policy(topic, workspace)
            except ValueError as e:
                result = str(e)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Sorry, I couldn't finish looking that u

## 1. Register the instructions and point `production` at them (Part 2, section 1)

The agent loads its instructions **by alias**, so promoting a new version later is a pointer
move, not a code change. v1 has three seeded problems: stale quick facts (30-day refunds), a
mandatory Terms-of-Service disclaimer, and "give your best guess".

In [27]:
import mlflow
from selfimprove import config

name = config.settings.prompt_name
v1 = mlflow.genai.register_prompt(
    name=name,
    template=open("examples/instructions.md").read(),
    commit_message="v1: initial instructions",
)
mlflow.genai.set_prompt_alias(name, alias="production", version=v1.version)

print(mlflow.genai.load_prompt(f"prompts:/{name}@production").template)

2026/10/28 12:18:44 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: UPDATE registered_models SET last_updated_time=? WHERE registered_models.workspace = ? AND registered_models.name = ?]
[parameters: (1793215124012, 'default', 'walkthrough-instructions')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)
Traceback (most recent call last):
  File "/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/.venv/lib/python3.11/site-packages/sqlalchemy/engine/base.py", line 1943, in _exec_single_context
    self.dialect.do_execute(
  File "/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/.venv/lib/python3.11/site-packages/sqlalchemy/engine/default.py", line 1180, in do_execute
    cursor.execute(statement, parameters)

MlflowException: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: UPDATE registered_models SET last_updated_time=? WHERE registered_models.workspace = ? AND registered_models.name = ?]
[parameters: (1793215124012, 'default', 'walkthrough-instructions')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

**Under the hood:** this cell is what the harness's `instructions.seed()` does (it also tags v1
`author=seed`). An alias is a movable pointer, so later the harness ships a new version by moving it,
and rolls back by moving it again.

**MLflow in this step**

* `mlflow.genai.register_prompt(name=..., template=..., commit_message=...)`: creates version 1 of the
  prompt in the Prompt Registry
* `mlflow.genai.set_prompt_alias(name, alias="production", version=...)`: points `production` at it
* `mlflow.genai.load_prompt("prompts:/<name>@production")`: resolves the alias to that version's text

In [ ]:
p = mlflow.genai.load_prompt(f"prompts:/{name}@production")
print(p.uri, "| version", p.version, "| aliases", p.aliases)
print("commit message:", p.commit_message)

prompts:/walkthrough-instructions/1 | version 1 | aliases ['production']
commit message: v1: initial instructions


## 2. One traced call

`selfimprove.agents.run()` opens an AGENT span, loads the instructions **inside** the trace (which
links the prompt version to the trace), tags the trace, and calls your agent. Everything your
agent does underneath (tool spans, autologged LLM calls) nests below it.

In [ ]:
from selfimprove import agents

print(inspect.getsource(agents.run))

def run(question: str, ref: str | int = "production", source: str = "eval", tags: dict | None = None) -> str:
    """Answer one question with the given instructions version. Produces one trace."""
    s = config.settings
    with mlflow.start_span(s.agent_name, span_type=SpanType.AGENT) as span:
        span.set_inputs({s.input_key: question})
        prompt = instructions.load(ref)  # loaded inside the trace: links prompt version <-> trace
        if mlflow.get_current_active_span() is not None:  # evaluate() probes predict_fn untraced once
            mlflow.update_current_trace(tags={
                "si_source": source, "si_prompt_version": str(prompt.version), "si_agent": s.agent,
                **(tags or {}),
            })
        text = prompt.template
        answer = adapter()(question, lessons.split(text)[0], workspace_for(text))
        span.set_outputs(answer)
        return answer



In [ ]:
answer = agents.run("How long can I wait to request a refund?", source="production")
print(answer)

Refunds are available within **14 days of purchase** on all plans (Monthly, Annual, Pro, and Enterprise). This replaced the earlier 30-day window on July 1, 2026, and applies to every purchase. My initial answer was based on outdated information, so I'm sorry for the confusion.

After the 14-day window, refunds aren't available, and no prorated refunds are issued.

To request a refund, go to **Settings → Billing → Request refund**.

Policies can vary by plan and region, so please double-check the Terms of Service for the details that apply to your account. If you're close to the deadline, I'd submit your request as soon as possible.


In [ ]:
mlflow.flush_trace_async_logging()   # traces are written in the background
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())
print(trace.info.tags["si_source"], "· instructions v" + trace.info.tags["si_prompt_version"])
nb.tree(trace)

production · instructions v1
my_agent  [AGENT]  3,829 ms
  in : {'question': 'How long can I wait to request a refund?'}
  out: Refunds are available within **14 days of purchase** on all plans (Monthly, Annual, Pro...
  litellm-completion  [LLM]  1,551 ms
    in : {'messages': [{'role': 'system', 'content': '# Acme Cloud Support Agent\n\nYou are the ...
    out: {'id': 'chatcmpl-c75422c7-a645-4e8d-aa1e-65aa03332277', 'created': 1793214302, 'model':...
  lookup_policy  [TOOL]  2 ms
    in : {'topic': 'billing', 'workspace': '/Users/jon/Documents/Claude/ODSC West 2026/self-impr...
    out: # Billing policy (Acme Cloud)  _Last updated: July 1, 2026_  ## Refunds - Refunds are a...
  litellm-completion  [LLM]  2,260 ms
    in : {'messages': [{'role': 'system', 'content': '# Acme Cloud Support Agent\n\nYou are the ...
    out: {'id': 'chatcmpl-6c9b680f-4359-4c15-aa3e-ccc81fd067e2', 'created': 1793214304, 'model':...


**Under the hood:** `agents.run()` makes exactly one trace per request. Spans are exported in the
background, which is why the cell above flushes before it reads the trace back.

**MLflow in this step**

* `mlflow.start_span(name="my_agent", span_type=SpanType.AGENT)`: opens the root span. With no trace
  active, that starts a new trace
* `mlflow.genai.load_prompt("prompts:/walkthrough-instructions@production")`: runs inside the span, so
  MLflow records the prompt version on the trace (the `mlflow.linkedPrompts` tag)
* `mlflow.update_current_trace(tags=...)`: adds `si_source`, `si_prompt_version` and `si_agent`
* `mlflow.flush_trace_async_logging()`: waits for spans still being exported
* `mlflow.get_trace(trace_id)`: reads the finished trace back

In [ ]:
print("trace id ", trace.info.trace_id, "| state", trace.info.state)
for k in ("si_source", "si_prompt_version", "si_agent", "mlflow.linkedPrompts"):
    print(f"{k:<21}", trace.info.tags.get(k))
print("spans    ", [(s.name, s.span_type) for s in trace.data.spans])

trace id  tr-ae39076a0e4021f052b442b2cb945ea1 | state OK
si_source             production
si_prompt_version     1
si_agent              python
mlflow.linkedPrompts  [{"name": "walkthrough-instructions", "version": "1"}]
spans     [('my_agent', 'AGENT'), ('litellm-completion', 'LLM'), ('lookup_policy', 'TOOL'), ('litellm-completion', 'LLM')]


Two tags matter for everything that follows:

* `si_prompt_version`: which version produced this answer, so signal can be sliced by version.
* `si_source`: `production` for real traffic, `eval` for evaluation, optimizer and CI calls. Only
  production traces ever count as signal, so an optimizer can never feed on its own output.

## 3. A week of traffic

`examples/questions.txt` holds the questions customers send. Replaying it twice gives the next
notebook something to look at. (`selfimprove replay examples/questions.txt` does the same from a
terminal.)

In [ ]:
questions = [q for q in open("examples/questions.txt").read().splitlines() if q and not q.startswith("#")]
for q in questions * 2:
    a = agents.run(q, source="production")
    print(f"{q[:55]:<55} -> {a[:70]!r}")

How long can I wait to request a refund?                -> 'Refunds are available within **14 days of purchase**. This applies to '
I'm on the annual plan. Do I get a longer refund window -> "No, annual customers don't get a longer refund window. Refunds are ava"
I bought Pro three weeks ago and barely used it. Can I  -> "I'm sorry, but I can't offer a refund for this purchase. Our current b"
How do I change the email address my invoices go to?    -> 'You can change the billing email in **Settings → Billing → Billing con'
What happens if my card payment fails?                  -> 'Policies can vary by plan and region, so please double-check the Terms'
How long is a password reset link valid?                -> 'Password reset links are valid for **24 hours**. If your link has expi'
I reset my password but still can't log in. What should -> "I'm sorry you're stuck. Policies can vary by plan and region, so pleas"
I typed my password wrong a bunch of times and now I'm  -> "I'm sorry you're

**Under the hood:** each call above wrote one more trace to the experiment. Every later step finds
production traffic with the same query. It is the filter the harness's `traces.production()` uses, and
why the `si_source` tag matters.

**MLflow in this step**

* `mlflow.search_traces(filter_string="tags.si_source = 'production'", flush=True)`: the production
  traces; `flush=True` waits for traces still being written in the background

In [ ]:
df = mlflow.search_traces(filter_string="tags.si_source = 'production'", flush=True)
print(len(df), "production traces in the store")
df.assign(version=df["tags"].map(lambda t: t.get("si_prompt_version")),
          question=df["request"].map(lambda r: r["question"][:50]))[["trace_id", "request_time", "version", "question"]].head(5)

29 production traces in the store


,trace_id,request_time,version,question
0,tr-306c14db35f9d03e6593c1d5bb1be28d,1793214411659,1,I lost my API key. Can you show it to me again?
1,tr-bd068364c32cea39b7ab9f9277c1413d,1793214408174,1,Does Acme Cloud integrate with Jira?
2,tr-af742d3256f5347f278f2b2767a604f4,1793214405541,1,How many teammates can I invite on Pro?
3,tr-4f67c4f2e9608f1fcf72d4307b04b457,1793214402940,1,How do I export all my data?
4,tr-160455a2b0f469120cf9be155dd3bec7,1793214398890,1,Can we set up SSO with Okta on our Pro plan?


Open the MLflow UI on this store to browse the traces. Run it from `harness/`: the `sqlite:///` path is
relative, so from anywhere else MLflow quietly creates a new, empty `mlflow.db` there instead. Use the
harness's own MLflow (`harness/.venv`, the same one as the `selfimprove (.venv)` kernel), not a global
`mlflow`, so the UI's version matches the store's schema:

```bash
cd harness
.venv/bin/mlflow ui --backend-store-uri sqlite:///.selfimprove/notebooks/mlflow.db --port 5003
```

Leave it running for the whole series: `nb.setup(fresh=True)` empties the store in place, so the UI
just shows the new, empty store after a reset.

**Next:** [01 · Signal](01_signal.ipynb), where people and judges say how these answers did.